# Telecom Churn Data Preprocessing & Feature Engineering

Goals:
- Clean and prepare the dataset for machine learning by addressing data quality issues.
- Handle anomalies, invalid values, and ensure consistent data types across features.
- Transform and encode categorical variables into model-compatible formats.
- Engineer new features (e.g., tenure-related metrics) to improve predictive power.
- Select and structure the final feature set for model training.
- Produce a clean, model-ready dataset to be used in the modeling stage.

In [25]:
import pandas as pd
import os
import plotly.express as px
from sklearn.preprocessing import OneHotEncoder 
from sklearn.preprocessing import StandardScaler


In [26]:
df = pd.read_csv("../../data/raw/telecom_churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [27]:
df.drop(columns=["customerID","TotalCharges","MultipleLines","SeniorCitizen","Partner"], inplace=True)
df.head()

,gender,Dependents,tenure,PhoneService,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,Churn
0,Female,No,1,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,No
1,Male,No,34,Yes,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,No
2,Male,No,2,Yes,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,Yes
3,Male,No,45,No,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,No
4,Female,No,2,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,Yes


In [28]:
services = [
"OnlineSecurity",
"OnlineBackup",
"DeviceProtection",
"TechSupport",
"StreamingTV",
"StreamingMovies"
]

df["num_services"] = (df[services] == "Yes").sum(axis=1)
df.drop(columns=services, inplace=True)

In [29]:
df["tenure_group"] = pd.cut(
    df["tenure"],
    bins=[0, 12, 24, 48, 72],
    labels=["new", "early", "established", "loyal"]
)

In [30]:
df["charge_level"] = pd.cut(
    df["MonthlyCharges"],
    bins=[0,40,80,120],
    labels=["low","medium","high"]
)

In [31]:
mapping_dict = { 'No': 0, 'Yes': 1}

df['Churn'] = df['Churn'].map(mapping_dict)

In [32]:
df.info()

df.head()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   gender            7043 non-null   str     
 1   Dependents        7043 non-null   str     
 2   tenure            7043 non-null   int64   
 3   PhoneService      7043 non-null   str     
 4   InternetService   7043 non-null   str     
 5   Contract          7043 non-null   str     
 6   PaperlessBilling  7043 non-null   str     
 7   PaymentMethod     7043 non-null   str     
 8   MonthlyCharges    7043 non-null   float64 
 9   Churn             7043 non-null   int64   
 10  num_services      7043 non-null   int64   
 11  tenure_group      7032 non-null   category
 12  charge_level      7043 non-null   category
dtypes: category(2), float64(1), int64(3), str(7)
memory usage: 619.5 KB


,gender,Dependents,tenure,PhoneService,InternetService,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,Churn,num_services,tenure_group,charge_level
0,Female,No,1,No,DSL,Month-to-month,Yes,Electronic check,29.85,0,1,new,low
1,Male,No,34,Yes,DSL,One year,No,Mailed check,56.95,0,2,established,medium
2,Male,No,2,Yes,DSL,Month-to-month,Yes,Mailed check,53.85,1,2,new,medium
3,Male,No,45,No,DSL,One year,No,Bank transfer (automatic),42.30,0,3,established,medium
4,Female,No,2,Yes,Fiber optic,Month-to-month,Yes,Electronic check,70.70,1,0,new,medium


In [33]:
categorical_cols = [        
"gender",
"Dependents",
"PhoneService",
"InternetService",
"Contract",
"PaperlessBilling",
"PaymentMethod",
"charge_level",
"tenure_group"
]
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
df.head()

,tenure,MonthlyCharges,Churn,num_services,gender_Male,Dependents_Yes,PhoneService_Yes,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,charge_level_medium,charge_level_high,tenure_group_early,tenure_group_established,tenure_group_loyal
0,1,29.85,0,1,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False
1,34,56.95,0,2,True,False,True,False,False,True,False,False,False,False,True,True,False,False,True,False
2,2,53.85,1,2,True,False,True,False,False,False,False,True,False,False,True,True,False,False,False,False
3,45,42.30,0,3,True,False,False,False,False,True,False,False,False,False,False,True,False,False,True,False
4,2,70.70,1,0,False,False,True,True,False,False,False,True,False,True,False,True,False,False,False,False


In [34]:
num_features = ["tenure","MonthlyCharges","num_services"]

# Initialize scaler
scaler = StandardScaler()

# Fit and transform
df[num_features] = scaler.fit_transform(df[num_features])

# Check scaled data
df[num_features].head()

,tenure,MonthlyCharges,num_services
0,-1.277445,-1.160323,-0.561776
1,0.066327,-0.259629,-0.020519
2,-1.236724,-0.362660,-0.020519
3,0.514251,-0.746535,0.520738
4,-1.236724,0.197365,-1.103033


In [35]:

output_dir = "../../data/processed/"
os.makedirs(output_dir, exist_ok=True)  

output_file = os.path.join(output_dir, "telecom_churn_clean.csv")
df.to_csv(output_file, index=False)